# 选修E2 · Day 3：营销组合优化：MMM、MTA与增量测量

> **真实库**：statsmodels + sklearn + scipy + causaldata
> **真实数据**：NSW (National Supported Work) 真实 RCT 数据，445 条真实样本 + 真实快消品 MMM 参数结构
> **核心任务**：MMM（Adstock+Ridge+贡献分解）+ MTA（马尔可夫链移除法）+ 增量测量（RCT+合成控制+DML）+ 预算优化

**学习目标**：
1. 用 statsmodels+sklearn 实现 MMM 全流程（Adstock 变换 + Ridge 回归 + 贡献分解）
2. 用 numpy+pandas 实现 MTA 马尔可夫链移除法 + 渠道功劳分配
3. 用 NSW 真实 RCT 数据做增量测量（朴素均值差 + 增量率 + ROI）
4. 用 numpy+pandas 实现合成控制（加权对照构造反事实 + ATT 估计）
5. 用 sklearn+statsmodels 实现 DML（双重机器学习交叉拟合 + 处理效应估计）
6. 用 scipy.optimize 基于MMM系数做预算优化（约束最大化）

## 环境准备：导入真实库

**真实库说明**：
- statsmodels：MMM 回归建模与统计推断
- sklearn：Ridge（MMM 共线性稳健）+ RandomForest（DML）+ StandardScaler + KFold
- scipy.optimize：预算优化数值求解
- causaldata：NSW 真实 RCT 数据
- pandas + numpy：数据处理与矩阵运算

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from scipy.optimize import minimize
from causaldata.nsw_mixtape import load_pandas

print('库导入完成：statsmodels + sklearn + scipy + causaldata')
import statsmodels; print(f'statsmodels version: {statsmodels.__version__}')
import sklearn; print(f'sklearn version: {sklearn.__version__}')
import scipy; print(f'scipy version: {scipy.__version__}')

## TODO1：MMM 拟合 - Adstock + Ridge + 贡献分解

**营销场景**：你是某快消品的营销分析师，需要用 MMM 评估四个渠道（search_ads/social_ads/display_ads/email_marketing）对周度销量的贡献，并做渠道贡献分解。

**MMM 核心模型**：
```
Sales_t = Base + Σ(βi × Adstock(Spend_i,t)) + Σ(γj × Control_j,t) + ε_t
Adstock_t = Spend_t + λ × Adstock_{t-1}
```

**真实快消品参数**（来自 Google Meridian / Meta Robyn 案例范围）：
- search_ads: λ=0.20 (即时响应), β=1.5 (强转化)
- social_ads: λ=0.40 (中周期), β=1.0 (品牌+转化)
- display_ads: λ=0.60 (长尾品牌), β=0.6 (品牌为主)
- email_marketing: λ=0.15 (触发式), β=1.2 (较高)

**任务**：
1. 生成 104 周营销数据（用真实快消品参数结构 + 季节性 + 节假日 + 竞品 + 噪声）
2. 实现 `apply_adstock(spend, decay)` 函数
3. 对四个渠道应用 Adstock 变换
4. 用 `StandardScaler + Ridge(alpha=1.0)` 拟合 MMM
5. 计算 R² 和渠道贡献分解（按反标准化系数 × 原始 X 均值归一化）
6. 打印 R²、各渠道系数、Adstock 衰减率、贡献分解

In [ ]:
# TODO1: MMM 拟合 - Adstock + Ridge + 贡献分解
# 提示：apply_adstock(spend, decay) 用循环; Ridge(alpha=1.0) 拟合; 贡献=反标准化系数×原始X均值
# TODO: 你的代码
raise NotImplementedError

# 1. 生成 104 周真实结构营销数据
# np.random.seed(42)
# weeks = 104
# search_ads = np.random.gamma(2, 5000, weeks) + 1000
# social_ads = np.random.gamma(2, 3000, weeks) + 800
# display_ads = np.random.gamma(2, 2000, weeks) + 500
# email_marketing = np.random.gamma(2, 1000, weeks) + 200
# seasonality = 1 + 0.3 * np.sin(2 * np.pi * np.arange(weeks) / 52)
# holiday_flag = np.isin(np.arange(weeks) % 52, [51, 0, 1, 11, 12]).astype(int)
# competitor_promo = (np.random.rand(weeks) < 0.2).astype(int)
# adstock_real = {'search_ads': 0.20, 'social_ads': 0.40, 'display_ads': 0.60, 'email_marketing': 0.15}
# beta_real = {'search_ads': 1.5, 'social_ads': 1.0, 'display_ads': 0.6, 'email_marketing': 1.2}
# base_sales = 50000
# sales = base_sales + Σ(βi × Adstock_i) + 5000*seasonality + 3000*holiday_flag - 2000*competitor_promo + noise

# 2. 实现 apply_adstock(spend, decay)

# 3. 对四个渠道应用 Adstock 变换

# 4. StandardScaler + Ridge(alpha=1.0) 拟合

# 5. 计算 R² 和贡献分解（反标准化系数 × 原始 X 均值）

# 6. 打印结果

## TODO2：MTA 马尔可夫链移除法

**营销场景**：你需要用 MTA 评估用户触点路径中每个渠道的功劳。用马尔可夫链移除法（Removal Effect）计算每个渠道对转化率的贡献。

**马尔可夫链移除法算法**：
1. 收集用户路径（这里用模拟生成 5000 条路径）
2. 构建一阶转移矩阵（Start / 各渠道 / Conversion / Null）
3. 计算基线转化率（Start -> Conversion 的吸收概率）
4. 对每个渠道 c：移除 c（将其所有出转移重定向到 Null），重新计算转化率
5. 移除效应 = (基线 - 移除后) / 基线
6. 归一化移除效应 -> 每个渠道 MTA 功劳分配

**任务**：
1. 生成 5000 条用户路径（4 个渠道 + Conversion/Null 终止态）
2. 构建一阶转移矩阵（用 pandas crosstab 计算转移概率）
3. 计算基线转化率（用矩阵幂或迭代法）
4. 实现 `compute_removal_effect(transition_matrix, channel)` 函数
5. 对 4 个渠道计算移除效应，归一化为功劳分配
6. 打印每个渠道的 MTA 功劳占比

In [ ]:
# TODO2: MTA 马尔可夫链移除法
# 提示：用 pd.crosstab 构建转移矩阵; 移除渠道=将该行所有出转移重定向到 Null
# TODO: 你的代码
raise NotImplementedError

# 1. 生成 5000 条用户路径
# channels = ['search', 'social', 'display', 'email']
# paths = []
# for _ in range(5000):
#     n_touch = np.random.randint(1, 5)
#     path = list(np.random.choice(channels, n_touch))
#     if np.random.rand() < 0.3:
#         path.append('Conversion')
#     else:
#         path.append('Null')
#     paths.append(path)

# 2. 构建 (current, next) 转移对，用 pd.crosstab 归一化得转移矩阵

# 3. 计算基线转化率（Start -> Conversion 概率）

# 4. 实现 compute_removal_effect(tm, channel)

# 5. 计算每个渠道移除效应，归一化

# 6. 打印 MTA 功劳分配

## TODO3：增量测量 - NSW RCT 朴素均值差 + 增量率 + ROI

**营销场景**：你需要测量某广告投放的真实增量。NSW RCT 数据是金标准--treat=1 表示收到广告曝光，re78 表示投放后销售。

**任务**：
1. 加载 NSW 数据（`causaldata.nsw_mixtape.load_pandas()`）
2. 计算 treated 和 control 的 re78 均值
3. 朴素均值差 = treated_mean - control_mean（RCT 下即真实增量 ATE）
4. 计算增量率 = 增量 / treated_mean
5. 假设广告投入 = 2000/人，计算增量 ROI = (增量 - 投入) / 投入
6. 用 `scipy.stats.ttest_ind` 做显著性检验
7. 打印所有结果

**关键解读**：
- RCT 下均值差 = 真因果效应（无混杂）
- 增量率 > 30%：广告创造新需求
- 增量率 < 10%：广告在收割已会购买的用户
- 增量 ROI > 0：广告值得投放

In [ ]:
# TODO3: NSW RCT 朴素均值差 + 增量率 + ROI
# 提示：treated=df[df.treat==1], control=df[df.treat==0]; ATE=treated.re78.mean()-control.re78.mean()
# TODO: 你的代码
raise NotImplementedError

# 1. 加载 NSW 数据

# 2. 计算 treated 和 control 的 re78 均值

# 3. 朴素均值差（ATE）

# 4. 增量率 = ATE / treated_mean

# 5. 增量 ROI（假设投入=2000/人）

# 6. t 检验

# 7. 打印结果

## TODO4：合成控制 - 加权对照构造反事实 + ATT 估计

**营销场景**：当 RCT 不可行时，用合成控制法构造"合成实验组"的反事实。NSW 数据虽是 RCT，但我们假装"无法做 RCT"，用控制组样本加权拟合实验组 pre-period（re74/re75），再用此权重在 post-period（re78）构造反事实。

**合成控制算法**：
1. 分离 treated 和 control 组
2. 用 re74/re75（pre-period）作为匹配变量
3. 求解权重 w：minimize ||treated_pre_mean - X_control.T @ w||²，约束 w >= 0, sum(w) = 1
4. 合成 re78 = X_control_post.T @ w（即 control 组 re78 的加权和）
5. ATT = treated_re78_mean - synthetic_re78
6. 与 TODO3 的 RCT 真值对比

**任务**：
1. 加载 NSW，分 treated / control
2. 构造 pre-period 特征矩阵 X（re74/re75）
3. 用 `scipy.optimize.minimize` 求解权重（约束 sum(w)=1, w>=0）
4. 用权重构造合成 re78
5. 计算 ATT = treated_re78 - synthetic_re78
6. 与 TODO3 的 ATE 对比，评估合成控制偏差

In [ ]:
# TODO4: 合成控制 - 加权对照构造反事实
# 提示：用 scipy.optimize.minimize 求 w, 约束 sum(w)=1, w>=0, 目标=pre-period 匹配
# TODO: 你的代码
raise NotImplementedError

# 1. 加载 NSW，分 treated / control

# 2. 构造 pre-period 特征 X（re74, re75）

# 3. 用 minimize 求解权重 w

# 4. 用 w 构造合成 re78

# 5. 计算 ATT

# 6. 与 TODO3 ATE 对比

## TODO5：DML 双重机器学习 - 交叉拟合 + 处理效应估计

**营销场景**：DML（Double Machine Learning, Chernozhukov 2018）是 2026 因果机器学习前沿。用 ML 拟合处理和结果的混杂模型，残差化后用 OLS 估计处理效应，可处理高维非线性混杂。

**DML 算法**（Cross-fitting + Double debiasing）：
1. 用 KFold 把数据分两半
2. 在 fold A 上训练 m(x)=E[T|X]（RandomForestRegressor），在 fold B 预测 T_hat
3. 在 fold A 上训练 g(x)=E[Y|X]（RandomForestRegressor），在 fold B 预测 Y_hat
4. 计算残差：T_tilde = T - T_hat, Y_tilde = Y - Y_hat
5. 用 OLS 回归 Y_tilde ~ T_tilde，系数即处理效应 θ_DML
6. 与 TODO3 的 RCT 真值对比

**任务**：
1. 加载 NSW，准备 X（age/educ/re74/re75 等）, T（treat）, Y（re78）
2. 用 KFold(n_splits=2) 做交叉拟合
3. 训练两个 RandomForestRegressor 拟合 T 和 Y
4. 计算残差 T_tilde, Y_tilde
5. 用 `sm.OLS` 回归 Y_tilde ~ T_tilde，提取系数 θ_DML
6. 与 TODO3 的 ATE 对比，评估 DML 偏差

In [ ]:
# TODO5: DML 双重机器学习
# 提示：KFold(2) 交叉拟合; RandomForest 拟合 T~X 和 Y~X; OLS 残差回归
# TODO: 你的代码
raise NotImplementedError

# 1. 准备 X (age/educ/black/hisp/marr/nodegree/re74/re75), T (treat), Y (re78)

# 2. KFold(2) 交叉拟合

# 3. RandomForest 拟合 T~X 和 Y~X

# 4. 计算残差 T_tilde, Y_tilde

# 5. OLS 回归 Y_tilde ~ T_tilde，提取系数

# 6. 与 TODO3 ATE 对比

## TODO6：预算优化 - scipy.optimize 约束最大化

**营销场景**：基于 TODO1 的 MMM 系数，在总预算 B 约束下，求每个渠道分配 x_i 最大化预测销量。

**优化模型**：
```
maximize  Base + Σ(βi × sqrt(Adstock(x_i, λi).mean()))
subject to Σ(x_i) = B, x_i >= 0
```

**注意**：sqrt 饱和效应（真实营销有边际递减）使优化器不会全押一个渠道。

**任务**：
1. 复用 TODO1 的 MMM 系数（βi 和 λi）
2. 定义目标函数 `neg_sales(allocation)`：给定分配，计算预测销量（取负号因 minimize）
3. 约束：`sum(allocation) = B`，边界 `0 <= x_i <= B`
4. 用 `scipy.optimize.minimize`（SLSQP 方法）求解
5. 与均匀分配对比（每个渠道 B/4），计算销量提升 %
6. 打印最优分配和销量提升

In [ ]:
# TODO6: 预算优化 - scipy.optimize
# 提示：objective 返回 -sales; 约束 sum=B; 用 method='SLSQP'; sqrt 饱和效应
# TODO: 你的代码
raise NotImplementedError

# 1. 复用 TODO1 的 MMM 系数（β 放大 30 倍演示，baseline 降低使渠道贡献占比显著）

# 2. 定义目标函数 neg_sales(allocation)（含 sqrt 饱和效应）

# 3. 约束 + 边界

# 4. minimize 求解（method='SLSQP', options={'maxiter':500, 'ftol':1e-9}）

# 5. 与均匀分配对比

# 6. 打印最优分配 + 销量提升 + 渠道效率

## 总结：MMM + MTA + 增量测量 -> 处方性营销组合优化

完成 6 个 TODO 后，你应能回答：

1. **MMM 贡献分解**：哪个渠道贡献最高？R² 多少？Adstock 衰减率是否符合业务直觉？
2. **MTA 功劳分配**：移除效应最高的渠道是哪个？与 MMM 贡献排名一致吗？
3. **增量测量**：NSW 的真实增量（ATE）多少？增量率多少？增量 ROI 正吗？
4. **合成控制**：合成控制的 ATT 与 RCT 真值偏差多少？为什么？
5. **DML**：DML 估计与 RCT 真值偏差多少？DML 在观测数据中的价值是什么？
6. **预算优化**：最优分配与均匀分配相比销量提升多少？为什么把更多预算给某渠道？

**关键认知**：
- 三大归因方法各有优缺点，最好的方案是组合使用：MMM 战略 + MTA 战术 + 增量验证
- RCT 是金标准，但实操中常不可行；合成控制 + DML 是观测数据下的因果测量工具
- 增量率比表面 ROAS 更重要--高 ROAS 不等于高增量
- 预算优化基于历史数据外推，需用增量测试验证关键决策
- 隐私时代（GDPR/CCPA/Cookie 消亡），MMM 和增量测试比 MTA 更有优势

**下一步**：本 Day 是选修 E2 的顶点。结合 Day 1 框架 + Day 2 CLV/流失 + Day 3 营销组合优化，形成完整的"描述 -> 诊断 -> 预测 -> 处方"营销分析闭环。